# FIFA World Cup 2026 — Tournament Prediction

End-to-end pipeline for the **48-team** format (12 groups of 4, 32 knockout entrants):

1. **Trains** Random Forest models on historical World Cup match scores (2002–2022, with Kaggle pre-tournament features)
2. **Predicts** the **72 group-stage** fixtures from `Data/clean_fifa_worldcup_fixture.csv`
3. **Builds** per-group standings (A–L) and qualifies **top two per group + 8 best third-placed** teams
4. **Simulates** the official Wikipedia knockout bracket (Round of 32 → Final + third-place match)

**Outputs:** `Data/fifa_worldcup_2026_predictions.csv`, `Data/fifa_worldcup_2026_standings.csv`, `Data/predicted_tournament_results.csv`, `Data/final_tournament_standings.csv`

Group membership comes from `Data/group_standings_2026.pkl` (`Table_Extraction.ipynb`). Third-place bracket slots use FIFA-eligible group lists from the fixture file; exact Annex C mapping for all 495 combinations is not modelled.

In [1]:
# Core libraries for data handling and score-prediction models
import sys

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

sys.path.insert(0, "scripts")
sys.path.insert(0, ".")
from kaggle_team_features import (
    default_kaggle_dir,
    enrich_matches,
    match_model_feature_columns,
    training_matches,
)
from tournament_2026 import (
    build_qualification_lookup,
    compute_group_standings,
    default_group_standings_path,
    fifa_ranks_from_predictions,
    group_standings_to_dataframe,
    load_group_teams,
    qualification_summary,
    rank_third_place_teams,
    simulate_knockout_bracket,
    split_fixtures,
)
import config
from config import adjust_coming_home_group_goals

/Users/nigelcuschieri/Development/world-cup-predictor/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Loaded from .env on import. Override here if needed for this session:
# config.ITS_COMING_HOME = True
print(f"ITS_COMING_HOME: {config.ITS_COMING_HOME}")
print(f"config file: {config.__file__}")
print(f".env exists: {(config.PROJECT_ROOT / '.env').exists()}")

ITS_COMING_HOME: True
config file: /Users/nigelcuschieri/Development/world-cup-predictor/config.py
.env exists: True


## Step 1 — Train score-prediction models

Load `Data/clean_fifa_worldcup_matches.csv` (Wikipedia scores + Kaggle features from `scripts/scrape_historical_matches.py`). Train on **2002+** rows with FIFA rank/points, 4-year form, host flags, squad market value (median-imputed when missing), squad age, World Cup titles, same-confederation matchups, and home-minus-away diffs.

In [3]:
# Historical World Cup results (re-run scrape script if Kaggle columns are missing)
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv").dropna(
    subset=["HomeGoals", "AwayGoals"]
)
if "SameContinent" not in df.columns:
    df = enrich_matches(df, default_kaggle_dir())

df_train = training_matches(df)

team_encoder = LabelEncoder()
team_encoder.fit(pd.concat([df_train["HomeTeam"], df_train["AwayTeam"]]).unique())
df_train = df_train.copy()
df_train["HomeTeamEncoded"] = team_encoder.transform(df_train["HomeTeam"])
df_train["AwayTeamEncoded"] = team_encoder.transform(df_train["AwayTeam"])

FEATURE_COLS = match_model_feature_columns()
X = df_train[FEATURE_COLS]
feature_imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(
    feature_imputer.fit_transform(X), columns=FEATURE_COLS, index=df_train.index
)

y_home = df_train["HomeGoals"]
y_away = df_train["AwayGoals"]

# 80/20 split for optional evaluation (models train on X_train below)
X_train, X_test, y_train_home, y_test_home = train_test_split(
    X, y_home, test_size=0.2, random_state=42
)
_, _, y_train_away, y_test_away = train_test_split(
    X, y_away, test_size=0.2, random_state=42
)

In [4]:
# Separate models let home/away scoring patterns differ
home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)

home_model.fit(X_train, y_train_home)
away_model.fit(X_train, y_train_away)


RandomForestRegressor(random_state=42)

## Step 2 — Predict 2026 group-stage fixtures

Apply the trained models to the **72 group-stage** matches only (knockout placeholders are resolved after qualification).

In [5]:
all_fixtures = pd.read_csv("Data/clean_fifa_worldcup_fixture.csv")
all_fixtures.rename(columns={"home": "HomeTeam", "away": "AwayTeam", "year": "Year"}, inplace=True)
group_fixtures, knockout_fixtures = split_fixtures(all_fixtures)

fixtures = enrich_matches(group_fixtures, default_kaggle_dir())
fixtures["HomeTeamEncoded"] = fixtures["HomeTeam"].apply(
    lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1
)
fixtures["AwayTeamEncoded"] = fixtures["AwayTeam"].apply(
    lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1
)

X_fixtures = pd.DataFrame(
    feature_imputer.transform(fixtures[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=fixtures.index,
)

# Round continuous predictions to whole goals for standings math
home_preds = np.round(home_model.predict(X_fixtures)).astype(int)
away_preds = np.round(away_model.predict(X_fixtures)).astype(int)
adjusted = [
    adjust_coming_home_group_goals(
        h, a, int(hg), int(ag), its_coming_home=config.ITS_COMING_HOME
    )
    for h, a, hg, ag in zip(
        fixtures["HomeTeam"], fixtures["AwayTeam"], home_preds, away_preds
    )
]
home_preds, away_preds = zip(*adjusted)
fixtures["PredictedHomeGoals"] = home_preds
fixtures["PredictedAwayGoals"] = away_preds

fixtures['Winner'] = np.select(
    [
        fixtures['PredictedHomeGoals'] > fixtures['PredictedAwayGoals'],
        fixtures['PredictedHomeGoals'] < fixtures['PredictedAwayGoals'],
    ],
    [fixtures['HomeTeam'], fixtures['AwayTeam']],
    default='Draw',
)


In [6]:
# One row per group-stage fixture with predicted scores and winner
fixtures.to_csv("Data/fifa_worldcup_2026_predictions.csv", index=False)
print(f"Predicted {len(fixtures)} group-stage fixtures ({len(knockout_fixtures)} knockout slots follow)")


Predicted 72 group-stage fixtures (32 knockout slots follow)


## Step 3 — Compute group standings & qualification

Rank teams **within each group** (A–L) by points, goal difference, and goals scored. Advance the top two from every group plus the **8 best third-placed** teams across all groups.

In [7]:
group_stage_results = pd.read_csv("Data/fifa_worldcup_2026_predictions.csv")
group_teams = load_group_teams(default_group_standings_path())

group_tables = compute_group_standings(group_stage_results, group_teams)
fifa_ranks = fifa_ranks_from_predictions(group_stage_results)
third_place_ranking = rank_third_place_teams(group_tables, fifa_ranks)
qualification_lookup = build_qualification_lookup(group_tables, third_place_ranking)
standings_df = qualification_summary(group_tables, third_place_ranking)

qualified = standings_df.loc[standings_df["Qualified"], "Team"].tolist()
print(f"\n🏆 {len(qualified)} teams qualified for the Round of 32:")
print(qualified)
print("\nBest third-placed teams:")
print(third_place_ranking.loc[third_place_ranking["Qualified"], ["Group", "Team", "Points", "GD", "GF"]])
print("\nSample group table (Group A):")
print(group_tables["Group A"])



🏆 32 teams qualified for the Round of 32:
['South Korea', 'Czech Republic', 'Mexico', 'Switzerland', 'Bosnia and Herzegovina', 'Canada', 'Brazil', 'Morocco', 'United States', 'Turkey', 'Germany', 'Ecuador', 'Ivory Coast', 'Netherlands', 'Japan', 'Sweden', 'Belgium', 'Iran', 'Spain', 'Uruguay', 'France', 'Norway', 'Senegal', 'Argentina', 'Algeria', 'Austria', 'Portugal', 'Colombia', 'DR Congo', 'England', 'Croatia', 'Ghana']

Best third-placed teams:
  Group         Team  Points  GD  GF
0     K     DR Congo       4   0   5
1     I      Senegal       4   0   4
2     J      Austria       4   0   4
3     B       Canada       4   0   4
4     E  Ivory Coast       4   0   4
5     F       Sweden       4   0   4
6     L        Ghana       4   0   4
7     A       Mexico       3   0   3

Sample group table (Group A):
     Group            Team  Points  GF  GA  GD  GroupRank
0  Group A     South Korea       5   4   3   1          1
1  Group A  Czech Republic       3   3   3   0          2
2  Grou

In [8]:
# Per-team qualification summary after the simulated group stage
standings_df.to_csv("Data/fifa_worldcup_2026_standings.csv", index=False)
group_standings_to_dataframe(group_tables).to_csv(
    "Data/fifa_worldcup_2026_group_tables.csv", index=False
)


## Step 4 — Simulate knockout stage

Retrain models on all historical teams, then play out the **official bracket** from `clean_fifa_worldcup_fixture.csv`: Round of 32 (16) → Round of 16 (8) → Quarterfinals (4) → Semifinals (2) → Third place → Final.

Ties in knockout matches are broken at random (no extra time / penalties modelled).

In [9]:
# Retrain on all 2002+ matches with Kaggle features
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv").dropna(
    subset=["HomeGoals", "AwayGoals"]
)
if "SameContinent" not in df.columns:
    df = enrich_matches(df, default_kaggle_dir())
df_train = training_matches(df)

team_encoder = LabelEncoder()
team_encoder.fit(pd.concat([df_train["HomeTeam"], df_train["AwayTeam"]]).unique())
df_train = df_train.copy()
df_train["HomeTeamEncoded"] = team_encoder.transform(df_train["HomeTeam"])
df_train["AwayTeamEncoded"] = team_encoder.transform(df_train["AwayTeam"])

X = df_train[FEATURE_COLS]
feature_imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(
    feature_imputer.fit_transform(X), columns=FEATURE_COLS, index=df_train.index
)
y_home = df_train["HomeGoals"]
y_away = df_train["AwayGoals"]

home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)
home_model.fit(X, y_home)
away_model.fit(X, y_away)


def encode_team(team):
    """Return encoded team ID, or -1 if the team never appeared in training data."""
    if team in team_encoder.classes_:
        return team_encoder.transform([team])[0]
    return -1


def knockout_match_features(home_team: str, away_team: str) -> pd.DataFrame:
    """Build imputed feature row for a 2026 knockout pairing (neutral-site style)."""
    row = pd.DataFrame(
        {"Year": [2026], "HomeTeam": [home_team], "AwayTeam": [away_team]}
    )
    row = enrich_matches(row, default_kaggle_dir())
    row["HomeTeamEncoded"] = encode_team(home_team)
    row["AwayTeamEncoded"] = encode_team(away_team)
    return pd.DataFrame(
        feature_imputer.transform(row[FEATURE_COLS]),
        columns=FEATURE_COLS,
    )

In [10]:
# Optional sanity check — full 2002+ training set used for knockout models
print(X.shape, y_home.shape, y_away.shape)


(384, 36) (384,) (384,)


In [11]:
def predict_knockout_goals(home_team: str, away_team: str) -> tuple[float, float]:
    match_features = knockout_match_features(home_team, away_team)
    return (
        float(home_model.predict(match_features)[0]),
        float(away_model.predict(match_features)[0]),
    )


results_df, knockout_standings_df, champion = simulate_knockout_bracket(
    knockout_fixtures,
    qualification_lookup,
    third_place_ranking,
    predict_knockout_goals,
    rng=np.random.default_rng(42),
    its_coming_home=config.ITS_COMING_HOME,
)

current_round = None
for _, match in results_df.iterrows():
    if match["Round"] != current_round:
        current_round = match["Round"]
        print(f"\n🔹 {current_round}:")
    print(
        f"⚽ Match {int(match['Match'])}: {match['Team1']} "
        f"({match['Predicted Goals Team1']:.1f}) vs {match['Team2']} "
        f"({match['Predicted Goals Team2']:.1f}) → Winner: {match['Winner']}"
    )

print(f"\n🏆🏆🏆 FIFA World Cup 2026 Champion: {champion} 🏆🏆🏆")

results_df.to_csv("Data/predicted_tournament_results.csv", index=False)
knockout_standings_df.to_csv("Data/final_tournament_standings.csv", index=False)

group_stage_standings = pd.read_csv("Data/fifa_worldcup_2026_standings.csv")
updated_standings = pd.concat([group_stage_standings, knockout_standings_df], ignore_index=True)
updated_standings.to_csv("Data/fifa_worldcup_2026_standings_Final_Updated.csv", index=False)



🔹 Round of 32:
⚽ Match 73: Czech Republic (1.8) vs Bosnia and Herzegovina (1.4) → Winner: Czech Republic
⚽ Match 74: Germany (2.4) vs Canada (0.9) → Winner: Germany
⚽ Match 75: Netherlands (1.9) vs Morocco (1.1) → Winner: Netherlands
⚽ Match 76: Brazil (1.8) vs Japan (1.3) → Winner: Brazil
⚽ Match 77: France (1.5) vs Sweden (0.8) → Winner: France
⚽ Match 78: Ecuador (1.2) vs Norway (1.5) → Winner: Norway
⚽ Match 79: South Korea (1.0) vs Senegal (1.4) → Winner: Senegal
⚽ Match 80: England (2.5) vs DR Congo (1.0) → Winner: England
⚽ Match 81: United States (1.3) vs Austria (1.2) → Winner: United States
⚽ Match 82: Belgium (1.9) vs Mexico (0.9) → Winner: Belgium
⚽ Match 83: Colombia (1.2) vs Croatia (1.5) → Winner: Croatia
⚽ Match 84: Spain (2.4) vs Algeria (0.9) → Winner: Spain
⚽ Match 85: Switzerland (1.3) vs Ivory Coast (1.6) → Winner: Ivory Coast
⚽ Match 86: Argentina (1.9) vs Uruguay (1.2) → Winner: Argentina
⚽ Match 87: Portugal (2.9) vs Ghana (1.2) → Winner: Portugal
⚽ Match 88: T